<a href="https://colab.research.google.com/github/martinhdezpacheco/tfg-scraping-madrid/blob/main/extraer_urls_redpiso.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
'''=======================================================
BLOQUE 0: Imports y configuración inicial
======================================================='''

# ATENCIÓN!!!!! Borrar los archivos de Colab

import requests
from bs4 import BeautifulSoup
import csv
import os
import time
from google.colab import files  # para subir/descargar archivos en Colab

# Cabecera para simular un navegador real (evita bloqueos básicos por user-agent vacío)
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

BASE_URL = "https://www.redpiso.es/venta-viviendas/madrid/madrid"
NOMBRE_CSV = "urls_recolectadas_redpiso.csv"
MAX_PAGINAS_SEGURIDAD = 150   # tope de seguridad, muy por encima de las páginas reales
PAUSA_ENTRE_PETICIONES = 1.5  # segundos de espera entre peticiones (cortesía con el servidor)

print("Bloque 0 completado: configuración cargada.")

Bloque 0 completado: configuración cargada.


In [3]:
'''=========================================================
BLOQUE 1: Subir el CSV de la sesión anterior (si ya existe)
========================================================='''

# Colab no tiene memoria entre sesiones, así que si ya recolectaste URLs antes,
# aquí las volvemos a cargar. Si es la primera vez, simplemente pulsa "Cancelar"
# en el diálogo de subida y el script empezará desde cero.

urls_previas = set()

print("Si ya tienes un urls_recolectadas_redpiso.csv de una sesión anterior, súbelo ahora.")
print("Si es la primera vez, pulsa 'Cancel upload' / cierra el diálogo.")

try:
    subido = files.upload()
    if NOMBRE_CSV in subido:
        with open(NOMBRE_CSV, "r", encoding="utf-8-sig", newline="") as f:
            reader = csv.DictReader(f, delimiter=";")
            for fila in reader:
                urls_previas.add(fila["url_detalle"])
        print(f"CSV cargado: {len(urls_previas)} URLs previas encontradas.")
    else:
        print("No se subió el archivo esperado. Empezamos desde cero.")
except Exception as e:
    print(f"No se subió ningún archivo ({e}). Empezamos desde cero.")

print(f"\nTotal de URLs previas cargadas: {len(urls_previas)}")

Si ya tienes un urls_recolectadas_redpiso.csv de una sesión anterior, súbelo ahora.
Si es la primera vez, pulsa 'Cancel upload' / cierra el diálogo.


No se subió el archivo esperado. Empezamos desde cero.

Total de URLs previas cargadas: 0


In [4]:
'''===========================================================================
BLOQUE 2: Recorrer todas las páginas de Redpiso y extraer las URLs de anuncio
==========================================================================='''

# Patrón de paginación confirmado:
#   Página 1 -> BASE_URL
#   Página N -> BASE_URL/pagina-N   (N >= 2)
# Cuando una página no tiene anuncios (0 encontrados), hemos llegado al final real.

def url_pagina(n):
    if n == 1:
        return BASE_URL
    return f"{BASE_URL}/pagina-{n}"


def extraer_urls_inmueble(html):
    soup = BeautifulSoup(html, "html.parser")
    urls = set()
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if "/inmueble/" in href:
            if href.startswith("/"):
                href = "https://www.redpiso.es" + href
            urls.add(href)
    return urls


urls_actuales = set()
pagina = 1

while pagina <= MAX_PAGINAS_SEGURIDAD:
    url = url_pagina(pagina)
    try:
        r = requests.get(url, headers=HEADERS, timeout=15)
    except Exception as e:
        print(f"Página {pagina} -> ERROR de conexión: {e}. Reintentando en 5s...")
        time.sleep(5)
        continue

    if r.status_code != 200:
        print(f"Página {pagina} -> status {r.status_code}. Detenemos aquí.")
        break

    urls_pagina = extraer_urls_inmueble(r.text)

    if len(urls_pagina) == 0:
        print(f"Página {pagina} -> 0 anuncios encontrados. Fin de la paginación real.")
        break

    urls_actuales |= urls_pagina
    print(f"Página {pagina} -> {len(urls_pagina)} anuncios | total acumulado: {len(urls_actuales)}")

    pagina += 1
    time.sleep(PAUSA_ENTRE_PETICIONES)

print(f"\nBloque 2 completado. URLs actuales encontradas en la web: {len(urls_actuales)}")

Página 1 -> 12 anuncios | total acumulado: 12
Página 2 -> 12 anuncios | total acumulado: 24
Página 3 -> 12 anuncios | total acumulado: 36
Página 4 -> 12 anuncios | total acumulado: 48
Página 5 -> 12 anuncios | total acumulado: 60
Página 6 -> 12 anuncios | total acumulado: 72
Página 7 -> 12 anuncios | total acumulado: 84
Página 8 -> 12 anuncios | total acumulado: 96
Página 9 -> 12 anuncios | total acumulado: 108
Página 10 -> 12 anuncios | total acumulado: 120
Página 11 -> 12 anuncios | total acumulado: 132
Página 12 -> 12 anuncios | total acumulado: 144
Página 13 -> 12 anuncios | total acumulado: 156
Página 14 -> 12 anuncios | total acumulado: 168
Página 15 -> 12 anuncios | total acumulado: 180
Página 16 -> 12 anuncios | total acumulado: 192
Página 17 -> 12 anuncios | total acumulado: 204
Página 18 -> 12 anuncios | total acumulado: 216
Página 19 -> 12 anuncios | total acumulado: 228
Página 20 -> 12 anuncios | total acumulado: 240
Página 21 -> 12 anuncios | total acumulado: 252
Página 22

In [5]:
'''=====================================================================
BLOQUE 3: Comparar lo recolectado ahora con lo que ya teníamos guardado
====================================================================='''

# La resta de conjuntos (set - set) nos da solo las URLs que no existían antes.

urls_nuevas = urls_actuales - urls_previas

print(f"URLs nuevas detectadas en esta sesión: {len(urls_nuevas)}")
if urls_nuevas:
    print("Ejemplo de URL nueva:", list(urls_nuevas)[0])

URLs nuevas detectadas en esta sesión: 778
Ejemplo de URL nueva: https://www.redpiso.es/inmueble/piso-en-venta-en-calle-de-san-narciso-canillejas-san-blas-madrid-RP292026155275


In [6]:
'''============================================
BLOQUE 4: Resumen numérico de la actualización
============================================'''

# (Recordatorio: nunca borramos URLs antiguas, aunque el anuncio ya no aparezca
# en la web -> las conservamos como observación histórica, como en Tecnocasa)

total_antes = len(urls_previas)
total_nuevas = len(urls_nuevas)
total_despues = total_antes + total_nuevas

print("=== RESUMEN DE LA ACTUALIZACIÓN ===")
print(f"URLs antes de esta sesión:   {total_antes}")
print(f"URLs nuevas encontradas:     {total_nuevas}")
print(f"URLs totales tras la unión:  {total_despues}")
print(f"(URLs vistas hoy en la web que ya conocíamos: {len(urls_actuales) - total_nuevas})")

=== RESUMEN DE LA ACTUALIZACIÓN ===
URLs antes de esta sesión:   0
URLs nuevas encontradas:     778
URLs totales tras la unión:  778
(URLs vistas hoy en la web que ya conocíamos: 0)


In [7]:
'''==================================================================
BLOQUE 5: Guardar el CSV final combinando URLs previas + URLs nuevas
=================================================================='''

# Formato: separador ';' y encoding 'utf-8-sig' (compatibilidad Excel español)

urls_finales = urls_previas | urls_nuevas  # unión de ambos conjuntos, sin duplicados

with open(NOMBRE_CSV, "w", encoding="utf-8-sig", newline="") as f:
    writer = csv.writer(f, delimiter=";")
    writer.writerow(["url_detalle"])
    for url in sorted(urls_finales):
        writer.writerow([url])

print(f"Bloque 5 completado: {NOMBRE_CSV} guardado con {len(urls_finales)} URLs totales.")

Bloque 5 completado: urls_recolectadas_redpiso.csv guardado con 778 URLs totales.


In [8]:
'''===================================================
BLOQUE 6: Descargar el CSV actualizado a tu ordenador
==================================================='''

# (Recuerda: descárgalo en su propia celda, sin descargas simultáneas,
# y no lo abras/guardes desde Excel para no cambiar el formato)

files.download(NOMBRE_CSV)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>